<a href="https://colab.research.google.com/github/IKKNIGHT/Protein_interactions/blob/main/Phase5_RFdiffusion_Runner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 5: run RFdiffusion on the design specs
Reads `design_specs/*.json` (from the Phase 3/4 notebook) and writes backbones to `Protein_interactions/rfdiff_outputs/<spec name>/`.
**Use an A100 runtime.**

Cell 2 installs RFdiffusion the **same way the working MYC notebook does** (`RFDiffusion_gnn.ipynb` in the MYC-PROTAC repo): straight into the Colab kernel python with pip, a DGL wheel cascade, and the repo's own `download_models.sh` for weights (3 x 484 MB, cached to Drive). No conda. **Cell 2 is CPU-only and that is correct** — the GPU is used only in Cell 3/4. First run installs + downloads (a few min); reruns skip both. If DGL can't install it falls back to SE3Transformer (slower but works).

In [ ]:
# ── Cell 1: Mount, settings, pick specs ───────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, glob, shutil, subprocess, time

BASE = '/content/drive/MyDrive/Protein_interactions'
SPEC_DIR, RFD_DRIVE = f'{BASE}/design_specs', f'{BASE}/rfdiff_outputs'
os.makedirs(RFD_DRIVE, exist_ok=True)

TARGET_FILTER = ['MDM2']                                   # Gate 1 = MDM2/p53 only
ARMS          = ['gnn', 'oracle', 'baseline', 'scrambled']
NUM_DESIGNS   = 30                                         # backbones per arm (~40 s each on an A100)

specs = [json.load(open(p)) for p in sorted(glob.glob(f'{SPEC_DIR}/*.json'))]
specs = [s for s in specs if s['target'] in TARGET_FILTER and s['arm'] in ARMS]
print(f'{len(specs)} specs selected:')
for s in specs: print(f"  {s['name']:<22} hotspots {s['hotspots']}  contig {s['contig']}")
assert specs, 'no specs found - run the Phase 3/4 notebook first'
subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader', shell=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
4 specs selected:
  5_1YCR_baseline        hotspots ['A109', 'A27', 'A104', 'A26', 'A47']  contig A25-109/0 40-60
  5_1YCR_gnn             hotspots ['A100', 'A99', 'A96', 'A54', 'A97']  contig A25-109/0 40-60
  5_1YCR_oracle          hotspots ['A96', 'A93', 'A94', 'A99', 'A73']  contig A25-109/0 40-60
  5_1YCR_scrambled       hotspots ['A46', 'A45', 'A44', 'A39', 'A56']  contig A25-109/0 40-60


CompletedProcess(args='nvidia-smi --query-gpu=name,memory.total --format=csv,noheader', returncode=0)

In [ ]:
import os, subprocess, shutil, time
WEIGHTS_DIR = '/content/RFdiffusion/models'; WEIGHTS_DRIVE = f'{BASE}/rfdiff_weights'
os.makedirs(WEIGHTS_DIR, exist_ok=True)
subprocess.run('apt-get -qq install -y aria2', shell=True, capture_output=True)

# nuke the truncated Base_ckpt.pt everywhere
for p in (f'{WEIGHTS_DIR}/Base_ckpt.pt', f'{WEIGHTS_DRIVE}/Base_ckpt.pt'):
    if os.path.exists(p):
        print(f'{os.path.getsize(p)/1e6:.0f} MB -> removing {p}'); os.remove(p)

HASHES = {'Base_ckpt.pt': '6f5902ac237024bdd0c176cb93063dc4',
          'Complex_base_ckpt.pt': 'e29311f6f1bf1af907f9ef9f44b8328b',
          'ActiveSite_ckpt.pt': '5532d2e1f3a4738decd58b19d633b3c3'}

def good(p): return os.path.exists(p) and os.path.getsize(p) > 450e6   # full file is ~484 MB

for f, h in HASHES.items():
    loc, drv = f'{WEIGHTS_DIR}/{f}', f'{WEIGHTS_DRIVE}/{f}'
    if good(loc):
        print(f'{f}: already good locally ({os.path.getsize(loc)/1e6:.0f} MB)')
    elif good(drv):
        print(f'{f}: restoring from Drive'); shutil.copy2(drv, loc)
    else:
        if os.path.exists(loc): os.remove(loc)
        print(f'{f}: downloading...', flush=True); t0 = time.time()
        subprocess.run(f'aria2c -x16 -s16 -k1M --console-log-level=warn --summary-interval=10 '
                       f'--allow-overwrite=true -d {WEIGHTS_DIR} -o {f} '
                       f'https://ipd.graylab.jhu.edu/RFdiffusion/{h}/{f}', shell=True)
        print(f'  {os.path.getsize(loc)/1e6:.0f} MB in {(time.time()-t0)/60:.1f} min')
    # re-cache to Drive only if the local file is full-size
    if good(loc) and not good(drv):
        shutil.copy2(loc, drv); print(f'  cached {f} to Drive')

print('\nfinal sizes:')
for f in HASHES:
    p = f'{WEIGHTS_DIR}/{f}'; print(f'  {f}: {os.path.getsize(p)/1e6:.0f} MB' if os.path.exists(p) else f'  {f}: MISSING')

484 MB -> removing /content/drive/MyDrive/Protein_interactions/rfdiff_weights/Base_ckpt.pt
Base_ckpt.pt: downloading...
  484 MB in 0.8 min
  cached Base_ckpt.pt to Drive
Complex_base_ckpt.pt: restoring from Drive
ActiveSite_ckpt.pt: restoring from Drive

final sizes:
  Base_ckpt.pt: 484 MB
  Complex_base_ckpt.pt: 484 MB
  ActiveSite_ckpt.pt: 484 MB


In [ ]:
# ── Cell 2: RFdiffusion in a Python 3.10 conda env (DGL has no py3.13 wheel; 3.10 is required) ──
import time, subprocess, sys, os, shutil; _t0 = time.time()
RFD, RFD_SCRIPT = '/content/RFdiffusion', '/content/RFdiffusion/scripts/run_inference.py'
WEIGHTS_DIR = f'{RFD}/models'; WEIGHTS = f'{WEIGHTS_DIR}/Complex_base_ckpt.pt'
WEIGHTS_DRIVE = f'{BASE}/rfdiff_weights'
os.makedirs(WEIGHTS_DIR, exist_ok=True); os.makedirs(WEIGHTS_DRIVE, exist_ok=True)
CONDA = '/opt/miniforge'; ENVPY = f'{CONDA}/envs/rfd/bin/python'

def sh(cmd, tail=2500, **kw):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw); return r.returncode, (r.stdout + r.stderr)[-tail:]

# clone (re-clone if incomplete)
if os.path.exists(RFD) and not os.path.exists(f'{RFD}/setup.py'): shutil.rmtree(RFD)
if not os.path.exists(RFD):
    print('cloning RFdiffusion...'); print(sh(f'git clone --depth 1 https://github.com/RosettaCommons/RFdiffusion.git {RFD}')[1][-200:])

ENV = dict(os.environ, PYTHONPATH=RFD, DGLBACKEND='pytorch')
def rfd_ok():
    rc, out = sh(f'{ENVPY} -c "import os; os.environ[\'DGLBACKEND\']=\'pytorch\'; import torch, dgl, hydra, e3nn, se3_transformer; '
                 f'from rfdiffusion.inference import utils; assert torch.cuda.is_available(); print(chr(82)+chr(70)+chr(68)+chr(79)+chr(75))"',
                 cwd=RFD, env=dict(ENV, PATH=f'{CONDA}/envs/rfd/bin:' + os.environ['PATH']))
    return (rc == 0 and any(l.strip()=='RFDOK' for l in out.splitlines())), out

if os.path.exists(ENVPY) and rfd_ok()[0]:
    print(f'  rfd env already works - skipping build  ({time.time()-_t0:.0f}s)')
else:
    if not os.path.exists(f'{CONDA}/bin/mamba'):
        print('installing Miniforge (brings mamba)...')
        print(sh('wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O /tmp/mf.sh && bash /tmp/mf.sh -b -p /opt/miniforge')[1][-200:])
    MB = f'{CONDA}/bin/mamba'
    print('creating py3.10 env with torch+cuda (mamba, ~2-3 min)...'); _ts = time.time()
    rc, out = sh(f'{MB} create -y -n rfd -c pytorch -c nvidia -c dglteam -c conda-forge '
                 f'python=3.10 pytorch=2.1.* pytorch-cuda=12.1 dgl -y', tail=3000)
    print(f'  env solve rc {rc} in {(time.time()-_ts)/60:.1f} min'); (rc and print(out[-1500:]))
    pip = f'{ENVPY} -m pip install -q'
    for step, cmd in [('deps', f'{pip} hydra-core omegaconf pyrsistent e3nn opt_einsum decorator icecream biopython scipy'),
                      ('SE3Transformer', f'{pip} --no-cache-dir {RFD}/env/SE3Transformer'),
                      ('RFdiffusion', f'{pip} --no-deps -e {RFD}')]:
        rc, out = sh(cmd); print(f'  {step}: {"OK" if rc==0 else "FAILED"}'); (rc and print(out[-1200:]))
    ok, out = rfd_ok(); print('  import check:', out.strip()[-250:])

# weights: restore from Drive (shell cp is more reliable than shutil on the Drive FUSE mount)
subprocess.run('apt-get -qq install -y aria2', shell=True, capture_output=True)
shutil.rmtree(WEIGHTS_DIR, ignore_errors=True); os.makedirs(WEIGHTS_DIR, exist_ok=True)
HASHES = {'Base_ckpt.pt':'6f5902ac237024bdd0c176cb93063dc4','Complex_base_ckpt.pt':'e29311f6f1bf1af907f9ef9f44b8328b','ActiveSite_ckpt.pt':'5532d2e1f3a4738decd58b19d633b3c3'}
good = lambda p: os.path.exists(p) and os.path.getsize(p) > 450e6
for f, h in HASHES.items():
    loc, drv = f'{WEIGHTS_DIR}/{f}', f'{WEIGHTS_DRIVE}/{f}'
    if good(drv):
        print(f'  {f}: copying from Drive...', flush=True)
        for attempt in range(3):
            subprocess.run(f'cp "{drv}" "{loc}"', shell=True)
            if good(loc): break
            print(f'    retry {attempt+1}'); time.sleep(3)
    if not good(loc):
        print(f'  {f}: downloading...', flush=True)
        if os.path.exists(loc): os.remove(loc)
        subprocess.run(f'aria2c -x16 -s16 -k1M --console-log-level=warn --summary-interval=10 --allow-overwrite=true -d {WEIGHTS_DIR} -o {f} https://ipd.graylab.jhu.edu/RFdiffusion/{h}/{f}', shell=True)
        if good(loc) and not good(drv): subprocess.run(f'cp "{loc}" "{drv}"', shell=True)
    print(f'    {f}: {os.path.getsize(loc)/1e6:.0f} MB' if os.path.exists(loc) else f'    {f}: MISSING')
print('  Complex_base_ckpt.pt:', round(os.path.getsize(WEIGHTS)/1e6) if os.path.exists(WEIGHTS) else 'MISSING', 'MB')

ok, out = rfd_ok()
print(f'\nRFdiffusion importable: {ok} | {out.strip()[-120:]}')
print(f'Cell 2 total: {(time.time()-_t0)/60:.1f} min')
assert ok and good(WEIGHTS), 'install/weights incomplete - paste output above'
PYBIN = ENVPY; ENV = dict(os.environ, PYTHONPATH=RFD, DGLBACKEND='pytorch', PATH=f'{CONDA}/envs/rfd/bin:'+os.environ['PATH'])
print('Using python:', PYBIN)

  rfd env already works - skipping build  (5s)
  Base_ckpt.pt: copying from Drive...
    Base_ckpt.pt: 484 MB
  Complex_base_ckpt.pt: copying from Drive...
    Complex_base_ckpt.pt: 484 MB
  ActiveSite_ckpt.pt: copying from Drive...
    ActiveSite_ckpt.pt: 484 MB
  Complex_base_ckpt.pt: 484 MB

RFdiffusion importable: True | ...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(enabled=False)
Cell 2 total: 0.2 min
Using python: /opt/miniforge/envs/rfd/bin/python


In [ ]:
# ── Cell 3: Smoke test - ONE design from the first spec ───────────────────────
def rfd_cmd(s, out_prefix, n, start=0):
    return (f"{PYBIN} {RFD_SCRIPT} inference.input_pdb={s['target_pdb']} inference.output_prefix={out_prefix} "
            f"inference.num_designs={n} inference.design_startnum={start} \"contigmap.contigs=['{s['contig']}']\" "
            f"ppi.hotspot_res=\"[{','.join(s['hotspots'])}]\" inference.ckpt_override_path={WEIGHTS} "
            f"denoiser.noise_scale_ca=0 denoiser.noise_scale_frame=0")
ENV = dict(os.environ, PYTHONPATH=RFD, DGLBACKEND='pytorch')

def run_stream(cmd, log_path):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=RFD, env=ENV)
    lines = []
    for line in p.stdout:
        lines.append(line)
        if any(k in line for k in ('Error', 'error', 'Traceback', 'Finished design', 'Making design')): print('   ', line.rstrip())
    p.wait(); open(log_path, 'w').writelines(lines)
    return p.returncode, ''.join(lines[-40:])

os.makedirs('/content/rfd_smoke', exist_ok=True)
t0 = time.time(); rc, tail = run_stream(rfd_cmd(specs[0], '/content/rfd_smoke/test', 1), '/content/rfd_smoke/log.txt')
pdbs = glob.glob('/content/rfd_smoke/test*.pdb')
print(f'smoke test rc={rc}, {len(pdbs)} pdb, {time.time()-t0:.0f}s')
if pdbs:
    from collections import OrderedDict
    ch = OrderedDict()
    for l in open(pdbs[0]):
        if l.startswith('ATOM') and l[12:16].strip() == 'CA': ch.setdefault(l[21], []).append(int(l[22:26]))
    print('chain layout (chain: n residues, first-last):', {c: (len(v), v[0], v[-1]) for c, v in ch.items()})
    print('expected: target residues keep original numbering, binder chain is LAST and numbered from 1')
else:
    print(tail)
assert pdbs, 'smoke test failed - paste the log tail above'

    [2026-10-15 17:33:37,021][__main__][INFO] - Making design /content/rfd_smoke/test_0
    [2026-10-15 17:34:22,145][__main__][INFO] - Finished design in 0.75 minutes
smoke test rc=0, 1 pdb, 75s
chain layout (chain: n residues, first-last): {'B': (49, 1, 49), 'A': (85, 25, 109)}
expected: target residues keep original numbering, binder chain is LAST and numbered from 1


In [ ]:
import glob
p = glob.glob('/content/rfd_smoke/test*.pdb')[0]
ch = {}
for l in open(p):
    if l.startswith('ATOM') and l[12:16].strip()=='CA':
        ch.setdefault(l[21], []).append(int(l[22:26]))
for c,v in ch.items():
    print(f'chain {c}: {len(v)} res, {min(v)}-{max(v)}, starts_at_1={min(v)==1}')
# also show the order chains first appear in the file
order=[]
for l in open(p):
    if l.startswith('ATOM') and l[21] not in order: order.append(l[21])
print('file order:', order, '| contig was:', specs[0]['contig'])

chain B: 49 res, 1-49, starts_at_1=True
chain A: 85 res, 25-109, starts_at_1=False
file order: ['B', 'A'] | contig was: A25-109/0 40-60


In [ ]:
# ── Cell 4: Run every selected spec (resumable; outputs copied to Drive after each spec) ──
summary = []
for s in specs:
    name = s['name']; loc = f'/content/rfdiff_outputs/{name}'; drv = f'{RFD_DRIVE}/{name}'; os.makedirs(loc, exist_ok=True); os.makedirs(drv, exist_ok=True)
    for f in glob.glob(f'{drv}/*'):                                   # restore anything already on Drive
        if not os.path.exists(f'{loc}/{os.path.basename(f)}'): shutil.copy2(f, loc)
    have = len(glob.glob(f'{loc}/design_*.pdb'))
    if have < NUM_DESIGNS:
        print(f'\n{name}: {have} done, running {NUM_DESIGNS - have} more'); t0 = time.time()
        rc, tail = run_stream(rfd_cmd(s, f'{loc}/design', NUM_DESIGNS - have, start=have), f'{loc}/run_{have}.log')
        print(f'  rc={rc} in {(time.time()-t0)/60:.1f} min')
        if rc: print(tail)
    for f in glob.glob(f'{loc}/design_*'):
        if f.endswith(('.pdb', '.trb')): shutil.copy2(f, drv)
    n = len(glob.glob(f'{drv}/design_*.pdb')); summary.append((name, n)); print(f'  {name}: {n} backbones on Drive')
print('\n', summary)
print('Next: run the Phase 6-8 notebook (ProteinMPNN + Boltz-2 + native comparison).')


5_1YCR_baseline: 0 done, running 30 more
    [2026-10-15 17:36:16,050][__main__][INFO] - Making design /content/rfdiff_outputs/5_1YCR_baseline/design_0
    [2026-10-15 17:37:00,987][__main__][INFO] - Finished design in 0.75 minutes
    [2026-10-15 17:37:00,987][__main__][INFO] - Making design /content/rfdiff_outputs/5_1YCR_baseline/design_1
    [2026-10-15 17:37:43,685][__main__][INFO] - Finished design in 0.71 minutes
    [2026-10-15 17:37:43,685][__main__][INFO] - Making design /content/rfdiff_outputs/5_1YCR_baseline/design_2
    [2026-10-15 17:38:25,852][__main__][INFO] - Finished design in 0.70 minutes
    [2026-10-15 17:38:25,852][__main__][INFO] - Making design /content/rfdiff_outputs/5_1YCR_baseline/design_3
    [2026-10-15 17:39:06,854][__main__][INFO] - Finished design in 0.68 minutes
    [2026-10-15 17:39:06,854][__main__][INFO] - Making design /content/rfdiff_outputs/5_1YCR_baseline/design_4
    [2026-10-15 17:39:49,246][__main__][INFO] - Finished design in 0.71 minutes
   